# ML CIE-2 – Activity 1: Dataset Preparation

Run each code cell from top to bottom in Google Colab. Keep the actual outputs and graphs generated by your run.

# ML CIE-2: Activity 1 and Activity 2
Google Colab notebook. Run all cells from top to bottom. Keep the real outputs, graphs and metric values generated by your run.

## Activity 1 – Dataset Preparation
Dataset: Titanic. Tasks: inspect data, check and handle missing values, remove duplicates, create at least two EDA visualizations, encode categorical features, scale suitable numeric features and save the prepared CSV.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
# Load Titanic dataset
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Inspect columns, data types and summary statistics
print("Columns:", df.columns.tolist())
df.info()
display(df.describe(include="all").T)

In [ ]:
# Missing values and percentages
missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})
display(missing_report)

In [ ]:
# Handle missing values, remove redundant columns and duplicates
df_clean = df.copy()
# deck has many missing values; alive duplicates survived; embark_town duplicates embarked
df_clean = df_clean.drop(columns=["deck", "alive", "embark_town"], errors="ignore")

if "age" in df_clean.columns:
    df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())

for col in df_clean.select_dtypes(include=["object", "category"]).columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print("Duplicate rows removed:", before - len(df_clean))
print("Shape after cleaning:", df_clean.shape)
display(df_clean.isnull().sum().to_frame("Missing Values"))

In [ ]:
# Basic text consistency: remove extra spaces and use lowercase
for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = df_clean[col].str.strip().str.lower()
display(df_clean.head())

### EDA: two required visualizations

In [ ]:
# Visualization 1: Age distribution
plt.figure(figsize=(8, 5))
sns.histplot(data=df_clean, x="age", bins=25, kde=True)
plt.title("Distribution of Passenger Age")
plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.show()

In [ ]:
# Visualization 2: Passenger count by class
plt.figure(figsize=(7, 5))
sns.countplot(data=df_clean, x="pclass")
plt.title("Passenger Count by Class")
plt.xlabel("Passenger Class")
plt.ylabel("Number of Passengers")
plt.show()

In [ ]:
# Encode categorical columns and scale numerical features
# Convert Boolean values to 0 and 1
bool_cols = df_clean.select_dtypes(include=["bool"]).columns
for col in bool_cols:
    df_clean[col] = df_clean[col].astype(int)

categorical_cols = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()
df_prepared = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=int)

# Scale numeric features, but do not scale the target column 'survived'
numeric_cols = df_prepared.select_dtypes(include=np.number).columns.tolist()
scale_cols = [c for c in numeric_cols if c != "survived"]
scaler = StandardScaler()
df_prepared[scale_cols] = scaler.fit_transform(df_prepared[scale_cols])

print("Encoded categorical columns:", categorical_cols)
print("Scaled numeric columns:", scale_cols)
print("Prepared shape:", df_prepared.shape)
display(df_prepared.head())

In [ ]:
# Save and download prepared CSV
output_file = "prepared_titanic.csv"
df_prepared.to_csv(output_file, index=False)
print("Saved:", output_file)
from google.colab import files
files.download(output_file)

### Activity 1 – Conclusion
The Titanic dataset was inspected and cleaned. Missing values were handled, duplicate rows were removed, two visualizations were generated, categorical features were encoded, numerical features were scaled, and the prepared data was saved as `prepared_titanic.csv`. Use the actual outputs from your Colab run in your submission.